# 03. BERT for Text Classification in Real-World NLP Tasks


## 1. What you will build

- A topic classification workflow with both a classical baseline and a BERT model.
- A reproducible train/validation/test split using external data files.
- A comparison table showing business tradeoffs between model families.


## 2. When to use this in real companies

Use this approach when label quality matters more than pure latency, especially for nuanced intents or heterogeneous text streams where classical models start to plateau. Keep the baseline in parallel for monitoring cost/performance tradeoffs.


## 3. Business goal

Classify incoming news-like documents into `world`, `sports`, `business`, and `sci_tech` categories with a robust model selection workflow.


## 4. Imports and setup


In [1]:
from pathlib import Path

import warnings

import numpy as np
import pandas as pd
import evaluate

from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Keep notebook output clean for teaching sessions.
warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data.dataloader")

from transformers.utils.notebook import NotebookProgressCallback

## 5. Load dataset from `data/03_data`


In [2]:
DATA_PATH = Path("../data/03_data/news_topic_synthetic_harder.csv")
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Classes: {df['label'].nunique()}")

df.head()

Rows: 1,200
Classes: 4


,article_id,split,text,label
0,BUS-TR-0001,train,"During a closely watched briefing, analysts sa...",business
1,BUS-TR-0002,train,According to people briefed on the discussions...,business
2,BUS-TR-0003,train,"In an update after the review, after reviewing...",business
3,BUS-TR-0004,train,According to people briefed on the discussions...,business
4,BUS-TR-0005,train,According to people briefed on the discussions...,business


## 6. Prepare splits and labels


In [3]:
label_names = sorted(df["label"].unique().tolist())
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

work_df = df.copy()
work_df["label_id"] = work_df["label"].map(label2id)

# Respetar el split fijo del dataset
train_full_df = work_df[work_df["split"] == "train"].copy()
test_df = work_df[work_df["split"] == "test"].copy()

# Sacar validación solo desde train
train_df, valid_df = train_test_split(
    train_full_df,
    test_size=0.15,
    random_state=42,
    stratify=train_full_df["label_id"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

Train: 748 | Valid: 132 | Test: 320


## 7. Classical baseline (TF-IDF + Logistic Regression)


In [4]:
baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
        ),
    ),
    ("logreg", LogisticRegression(max_iter=2000)),
])

baseline.fit(train_df["text"], train_df["label_id"])
b_preds = baseline.predict(test_df["text"])
b_acc = accuracy_score(test_df["label_id"], b_preds)
b_f1 = f1_score(test_df["label_id"], b_preds, average="weighted")

print(f"Baseline accuracy: {b_acc:.3f}")
print(f"Baseline weighted F1: {b_f1:.3f}")
print(classification_report(test_df["label_id"], b_preds, target_names=label_names, digits=3))

Baseline accuracy: 0.819
Baseline weighted F1: 0.820
              precision    recall  f1-score   support

    business      0.985     0.800     0.883        80
    sci_tech      0.795     0.825     0.810        80
      sports      0.744     0.762     0.753        80
       world      0.789     0.887     0.835        80

    accuracy                          0.819       320
   macro avg      0.828     0.819     0.820       320
weighted avg      0.828     0.819     0.820       320



## 8. Build Hugging Face datasets


In [5]:
def to_hf_dataset(frame: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(
        frame[["text", "label_id"]].rename(columns={"label_id": "label"}),
        preserve_index=False,
    )

train_ds = to_hf_dataset(train_df)
valid_ds = to_hf_dataset(valid_df)
test_ds = to_hf_dataset(test_df)

train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 748
})

## 9. Fine-tune DistilBERT

Note: when loading `distilbert-base-uncased` into a sequence-classification model, you may see `UNEXPECTED` and `MISSING` weights. This is expected: the masked-language-model head is discarded and a new classification head is initialized for downstream training.

This configuration is intentionally lightweight for teaching purposes.


BERT (Bidirectional Encoder Representations from Transformers):
- is a transformer-based language model. 
- it is used to understand text by taking into account the full context of a word within a sentence, looking at both the left and right sides.

Example:
In “Apple released a new phone,” Apple refers to a company.
In “I ate an apple,” apple refers to a fruit.

BERT **captures this difference through context.**

<h2 style="color:#f8fafc; font-family:Arial, sans-serif; margin-bottom: 16px;">
  TF-IDF + Cosine Similarity vs TF-IDF + Logistic Regression vs BERT
</h2>

<table style="width:100%; border-collapse:collapse; font-family:Arial, sans-serif; background:#111827; color:#e5e7eb;">
  
  <tr style="background:#2563eb; color:white;">
    <th style="padding:12px; border:1px solid #374151; text-align:left;">Aspect</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">TF-IDF + Cosine Similarity</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">TF-IDF + Logistic Regression</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">BERT</th>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Definition</strong></td>
    <td style="padding:12px; border:1px solid #374151;">
      A similarity-based method that compares TF-IDF vectors and assigns the label of the most similar training document.
    </td>
    <td style="padding:12px; border:1px solid #374151;">
      A supervised classification approach that uses TF-IDF features and Logistic Regression to learn decision boundaries.
    </td>
    <td style="padding:12px; border:1px solid #374151;">
      A transformer-based language model that generates contextual embeddings and can be fine-tuned for classification tasks.
    </td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Type</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Instance-based / retrieval approach</td>
    <td style="padding:12px; border:1px solid #374151;">Supervised machine learning model</td>
    <td style="padding:12px; border:1px solid #374151;">Deep learning (transformer-based)</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Feature representation</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Sparse TF-IDF vectors</td>
    <td style="padding:12px; border:1px solid #374151;">Sparse TF-IDF vectors</td>
    <td style="padding:12px; border:1px solid #374151;">Dense contextual embeddings</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Context understanding</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Limited (bag-of-words + n-grams)</td>
    <td style="padding:12px; border:1px solid #374151;">Limited (bag-of-words + n-grams)</td>
    <td style="padding:12px; border:1px solid #374151;">Strong (bidirectional context)</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Training</strong></td>
    <td style="padding:12px; border:1px solid #374151;">No explicit training (similarity matching)</td>
    <td style="padding:12px; border:1px solid #374151;">Requires supervised training</td>
    <td style="padding:12px; border:1px solid #374151;">Pretrained + fine-tuning required</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Performance</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Baseline performance</td>
    <td style="padding:12px; border:1px solid #374151;">Strong for many tasks</td>
    <td style="padding:12px; border:1px solid #374151;">State-of-the-art in many NLP tasks</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Speed</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Fast</td>
    <td style="padding:12px; border:1px solid #374151;">Fast</td>
    <td style="padding:12px; border:1px solid #374151;">Slow (relative to TF-IDF methods)</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Computational cost</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Low</td>
    <td style="padding:12px; border:1px solid #374151;">Low</td>
    <td style="padding:12px; border:1px solid #374151;">High</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Interpretability</strong></td>
    <td style="padding:12px; border:1px solid #374151;">High (direct similarity)</td>
    <td style="padding:12px; border:1px solid #374151;">Medium (feature weights)</td>
    <td style="padding:12px; border:1px solid #374151;">Low (black-box model)</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Best use case</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Retrieval, duplicate detection, similarity search</td>
    <td style="padding:12px; border:1px solid #374151;">Text classification, spam detection, categorization</td>
    <td style="padding:12px; border:1px solid #374151;">Advanced NLP tasks requiring deep contextual understanding</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Output</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Similarity score + nearest label</td>
    <td style="padding:12px; border:1px solid #374151;">Predicted class + probabilities</td>
    <td style="padding:12px; border:1px solid #374151;">Predicted class + contextual embeddings</td>
  </tr>

</table>

<p style="margin-top:16px; color:#9ca3af; font-family:Arial;">
  In summary, TF-IDF + Cosine Similarity provides a simple and interpretable baseline, TF-IDF + Logistic Regression offers a strong and efficient supervised solution, while BERT delivers superior contextual understanding at the cost of higher computational complexity.
</p>

In [6]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize_batch, batched=True)
valid_tok = valid_ds.map(tokenize_batch, batched=True)
test_tok = test_ds.map(tokenize_batch, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_weighted": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="artifacts/03_bert_text_classification",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=25,
    report_to="none",
    dataloader_pin_memory=False,
    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
bert_eval = trainer.predict(test_tok).metrics
bert_eval


Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.263044,0.067162,1.000000,1.000000
2,0.035734,0.027763,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'test_loss': 0.11758039891719818,
 'test_accuracy': 0.9875,
 'test_f1_weighted': 0.9875530419166708,
 'test_runtime': 25.5145,
 'test_samples_per_second': 12.542,
 'test_steps_per_second': 1.568}

## 10. Compare baseline vs BERT


In [7]:
bert_acc = bert_eval.get("test_accuracy", bert_eval.get("eval_accuracy"))
bert_f1 = bert_eval.get("test_f1_weighted", bert_eval.get("eval_f1_weighted"))

comparison = pd.DataFrame(
    {
        "model": ["TF-IDF + Logistic Regression", "DistilBERT fine-tuned"],
        "accuracy": [b_acc, bert_acc],
        "weighted_f1": [b_f1, bert_f1],
    }
)
comparison

,model,accuracy,weighted_f1
0,TF-IDF + Logistic Regression,0.81875,0.820239
1,DistilBERT fine-tuned,0.98750,0.987553


## 11. Inference examples


In [8]:
sample_texts = [
    "The board approved a merger after strong quarterly guidance.",
    "The forward scored a hat-trick in the semifinal.",
    "Scientists unveiled a new low-power semiconductor architecture.",
]

enc = tokenizer(sample_texts, return_tensors="pt", truncation=True, padding=True)
outputs = trainer.model(**enc).logits.detach().cpu().numpy()
pred_ids = outputs.argmax(axis=-1)

pd.DataFrame({
    "text": sample_texts,
    "predicted_label": [id2label[i] for i in pred_ids],
})

,text,predicted_label
0,The board approved a merger after strong quart...,business
1,The forward scored a hat-trick in the semifinal.,sports
2,Scientists unveiled a new low-power semiconduc...,sci_tech


## 12. Summary

- Data is externalized under `data/03_data` for reproducibility and teaching.
- The notebook compares a cheap baseline against a transformer model using the same split.
- This setup helps teams decide when BERT is worth the additional cost.
